# ការកក់សណ្ឋាគារជាមួយមីដលវែរ Priority Member

សៀវភៅកំណត់ត្រានេះបង្ហាញពី **មីដលវែរ​តាមមុខងារ** ដោយប្រើ Microsoft Agent Framework។ យើងកសាងលើឧទាហរណ៍នៃដំណើរការការងារជាភាពលក្ខណៈលក្ខខណ្ឌដោយបន្ថែមស្រទាប់មីដលវែរដែលផ្តល់អំណោយផលពិសេសសម្រាប់សមាជិកដែលមានអាទិភាព។

## អ្វីដែលអ្នកនឹងរៀន៖
1. **មីដលវែរ​ផ្អែកលើមុខងារ**៖ ឆក់ចូល និងកែប្រែលទ្ធផលមុខងារ
2. **ការចូលប្រើ context**៖ អាននិងកែប្រែ `context.result` បន្ទាប់ពីដំណើរការ
3. **ការអនុវត្តច្បាប់អាជីវកម្ម**៖ អត្ថប្រយោជន៍សមាជិកអាទិភាព
4. **ប្តូរលទ្ធផល**៖ ផ្លាស់ប្តូរពលកម្មមុខងារតាមស្ថានភាពអ្នកប្រើ
5. **ដំណើរការដដែល ប៉ុន្តែមានលទ្ធផលខុសគ្នា**៖ ការផ្លាស់ប្តូរសฤព្ធតាមមីដលវែរ

## សArchitecture​ Workflow ជាមួយមីដលវែរ៖

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## ភាពខុសគ្នាគន្លងពី Workflow ជាភាពលក្ខណៈលក្ខខណ្ឌ៖

**គ្មានមីដលវែរ** (14-conditional-workflow.ipynb):
- ភារីសគ្មានបន្ទប់ → ផ្ញើទៅ alternative_agent

**មានមីដលវែរ** (សៀវភៅកំណត់ត្រានេះ):
- អ្នកប្រើធម្មតា + ភារីស → គ្មានបន្ទប់ → ផ្ញើទៅ alternative_agent
- អ្នកប្រើអាទិភាព + ភារីស → 🌟 មីដលវែរ​ច្រានចោល! → មានបន្ទប់ → ផ្ញើទៅ booking_agent

## ការត្រូវការជាមុន៖
- បានដំឡើង Microsoft Agent Framework
- យល់ដឹងពីដំណើរការជាភាពលក្ខណៈលក្ខខណ្ឌ (មើល 14-conditional-workflow.ipynb)
- ពាក្យសម្ងាត់ GitHub ឬក៏កូនសោ API OpenAI
- ចំណេះដឹងមូលដ្ឋានអំពីគំរូមីដលវែរ


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ជំហាន ១៖ កំណត់ម៉ូដែល Pydantic សម្រាប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ

ម៉ូដែលទាំងនេះកំណត់ **ស្កីម៉ា** ដែលភ្នាក់ងារនឹងត្រឡប់មកវិញ។ យើងបានបន្ថែមវាល `priority_override` ដើម្បីតាមដានពេលដែល middleware លៃតម្រូវលទ្ធផលភាពអាចប្រើបាន។


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ជំហ៊ានទី 2៖ កំណត់ ទិន្នន័យសមាជិកអាទិភាព

សម្រាប់ការបង្ហាញនេះ យើងនឹងធ្វើការសម្ដែងមូលដ្ឋានទិន្នន័យសមាជិកអាទិភាព។ នៅក្នុងការផលិត វានឹងស្នើសុំទៅកាន់មូលដ្ឋានទិន្នន័យពិត ឬ API។

**សមាជិកអាទិភាព៖**
- `alice@example.com` - សមាជិក VIP
- `bob@example.com` - សមាជិក Premium  
- `priority_user` - គណនីសាកល្បង


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ជំហានទី ៣៖ បង្កើតឧបករណ៍កក់សណ្ឋាគារ

ដូចជាក្នុងសេចកី្តប្រតិបត្តិការជាលក្ខណៈលក្ខខណ្ឌផងដែរ ប៉ុន្តែពេលនេះវានឹងត្រូវបានទប់ស្តាប់ដោយ middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ជំហានទី ៤: 🌟 បង្កើត Priority Check Middleware (មុខងារស្នេញសំខាន់!)

នេះគឺជាមុខងារស្នេញរបស់សៀវភៅកំណត់នេះ។ Middleware៖

1. **ចាប់យក** ការហៅ​មុខងារ hotel_booking
2. **អនុវត្ត** មុខងារ​ដដែល​ដោយហៅ `next(context)`
3. **ពិនិត្យ​មើល** លទ្ធផល​នៅ​ក្នុង `context.result`
4. **ប្តូរប្រែក** លទ្ធផល ប្រសិនបើអ្នកប្រើប្រាស់មានអាទិភាព និងបន្ទប់មិនមាន
5. **ដាក់បញ្ចូន** លទ្ធផលដែលបានកែប្រែក្រឡៅទៅឱ្យភ្នាក់ងារ

**លំនាំសំខាន់៖**
```python
async def my_middleware(context, next):
    await next(context)  # ដំណើរការប្រតិបត្តិការ
    # ឥឡូវនេះ context.result មានលទ្ធផលនៃប្រតិបត្តិការ
    if some_condition:
        context.result = new_value  # ជំនួស!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## ជំហានទី ៥៖ កំណត់មុខងារជាលក្ខណៈសម្រាប់ការបញ្ជូន

មុខងារជាលក្ខណៈដូចគ្នានឹងលំហូរការងារជាលក្ខណៈ - ពួកវាស៊ើបអង្កេតលទ្ធផលដែលមានរចនាសម្ព័ន្ធដើម្បីកំណត់ការបញ្ជូន។


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## ជំហានទី ៦៖ បង្កើតកម្មវិធីអនុវត្តបង្ហាញផ្ទាល់ខ្លួន

កម្មវិធីអនុវត្តដូចមុន - បង្ហាញលទ្ធផលសកម្មភាពចុងក្រោយ។


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## ជំហានទី ៧៖ បញ្ចូលអថេរបរិបទបរិស្ថាន

កំណត់រចនា​ទីភ្នាក់ងារ LLM (Microsoft Foundry / Azure OpenAI)។


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ជំហាន 8: បង្កើតភ្នាក់ងារប្រាជ្ញាសិប្បនិម្មិតជាមួយ Middleware

**ភាពខុសគ្នាសំខាន់:** នៅពេលបង្កើត availability_agent យើងផ្ញើប៉ារ៉ាម៉ែត្រ `middleware`!

វាជាវិធីដែលយើងបញ្ចូល priority_check_middleware ចូលទៅក្នុងបណ្តារបញ្ចូលអំពីការហៅមុខងាររបស់ភ្នាក់ងារ។


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ជំហានទី ៩៖ បង្កើតដំណើរការ

រចនាសម្ព័ន្ធដំណើរការតែមួយដដែលដូចមុន - ការបញ្ជូនដែលមានលក្ខខណ្ឌដោយផ្អែកលើការចេញប្រាក់មាន។


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## ជំហានទី 10: ករណីសាកល្បង 1 - អ្នកប្រើប្រាស់ធម្មតា​នៅទីក្រុងប៉ារីស (មិនមានការលើសលុប)

អ្នកប្រើប្រាស់ធម្មតាយោងតាមកក់ប៉ារីស → គ្មានបន្ទប់ → កំណត់ផ្លូវទៅ alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ជំហាន 11: ករណីសាកល្បង 2 - 🌟 អ្នកប្រើប្រាស់អាទិភាពនៅប៉ារីស (មានការបដិសេធ!)

សមាជិកអាទិភាពព្យាយាមកំណត់កក់ប៉ារីស → គ្មានបន្ទប់ ban đầu → 🌟 មគ្គុទេសក៍បដិសេធ! → ផ្លូវទៅ booking_agent

**នេះជាការបង្ហាញសំខាន់នៃអំណាច middleware!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ជំហានទី 12៖ ករណីតេស្តទី 3 - អ្នកប្រើប្រាស់អាទិភាពនៅ Stockholm (មានស្រេចហើយ)

អ្នកប្រើប្រាស់អាទិភាពព្យាយាម Stockholm → បន្ទប់មាននៅ → មិនចាំបាច់បដិសេធ → ធ្វើ​ឲ្យ​ទៅ​កាន់ booking_agent

នេះបង្ហាញថា middleware ប្រតិបត្តិមួយតែលើកពេលចាំបាច់តែប៉ុណ្ណោះ!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## ចំណុចសំខាន់ៗ និង គំនិត Middleware

### ✅ អ្វីដែលអ្នកបានរៀន៖

#### **1. លំនាំ Middleware មូលដ្ឋានលើមុខងារ**

Middleware ចាប់ការហៅមុខងារដោយប្រើមុខងារ async ងាយៗមួយ:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # មុនការប្រតិបត្តិការមុខងារ
    print("Intercepting...")
    
    # ប្រតិបត្តិមុខងារ
    await next(context)
    
    # បន្ទាប់ពីការប្រតិបត្តិការមុខងារ - ពិនិត្យលទ្ធផល
    if context.result:
        # កែប្រែលទ្ធផលប្រសិនបើគឺចាំបាច់
        context.result = modified_value
```

#### **2. ការចូលប្រើ Context និង ការប្តូរវិលតប**

- `context.function` - ចូលប្រើមុខងារដែលត្រូវបានហៅ
- `context.arguments` - អានក្រោមមុខងារ
- `context.kwargs` - ចូលប្រើប៉ារ៉ាម៉ែត្រ​បន្ថែម
- `await next(context)` - ដំណើរការមុខងារ
- `context.result` - អាន/កែប្រែលទ្ធផលមុខងារ

#### **3. ការអនុវត្តផ្នែក Logic វិបិដាន**

Middleware របស់យើងអនុវត្តអត្ថប្រយោជន៍សមាជិកអាទិភាព៖
- **អ្នកប្រើប្រាស់ទូទៅ**: គ្មានការផ្លាស់ប្តូរ, ប្រព័ន្ធធម្មតា
- **អ្នកប្រើប្រាស់អាទិភាព**: ប្តូរ "គ្មានរបស់" → "មាន"
- **លក្ខខ័ណ្ឌវិបិដាន**: មិនប្តូរទេតែពេលដែលចាំបាច់

#### **4. ធ្វើដំណើរការដូចគ្នា ប៉ុន្តែលទ្ធផលខុសគ្នា**

មហិមាភាពនៃ middleware:
- ✅ គ្មានការផ្លាស់ប្តូរទៅលើរចនាសម្ព័ន្ធប្រតិបត្តិការ
- ✅ គ្មានការផ្លាស់ប្តូរទៅលើមុខងារ tools
- ✅ គ្មានការផ្លាស់ប្តូរទៅលើ logic ការបំពេញលក្ខខ័ណ្ឌ
- ✅ មាន middleware រួច→ មានអាកប្បកិរិយាផ្សេងគ្នា!

### 🚀 ការប្រើប្រាស់ពិតប្រាកដនៅពិភពលោក៖

1. **លក្ខណៈពិសេស VIP/Premium**
   - ប្តូរព្រំដែនអត្រាសម្រាប់អ្នកប្រើប្រាស់បូកប្រាក់
   - ផ្តល់ការចូលដំណើរការជាអាទិភាពទៅដើមទុន
   - បើកចំណុចពិសេសដោយ δυναμικό

2. **ការប្រឡង A/B**
   - ផ្លូវអ្នកប្រើទៅអនុវត្តន៍ខុសគ្នា
   - សាកល្បងលក្ខណៈពិសេសថ្មីជាមួយអ្នកប្រើជាក់លាក់
   - បញ្ចេញការប្រកាសលក្ខណៈពិសេសយ៉ាងតិចតួច

3. **សន្តិសុខ និងការអនុវត្តច្បាប់**
   - ត្រួតពិនិត្យការហៅមុខងារ
   - បិទបាំងប្រតិបត្ដិការប្រញាប់ប្រញាល់
   - អនុវត្តន៍ច្បាប់អាជីវកម្ម

4. **បំលែងប្រសិទ្ធភាព**
   - កំណត់ចុះបញ្ជីលទ្ធផលសម្រាប់អ្នកប្រើជាក់លាក់
   - ថតចាំពីប្រតិបត្ដិការចំណាយថ្លៃ
   - កំណត់ធនធាន δυναμικά

5. **ការកាន់កាប់កំហុស និងសាកល្បងឡើងវិញ**
   - ចាប់កំហុស និងដោះស្រាយយ៉ាងរលូន
   - អនុវត្តន៍ logic សាកល្បងឡើងវិញ
   - ត្រឡប់ទៅអនុវត្តខាងផ្សេង

6. **កំណត់ហេតុ និងត្រួតពិនិត្យ**
   - តាមដានពេលវេលាប្រតិបត្ដិមុខងារ
   - កំណត់ការផ្ទៀងផ្ទាត់ប៉ារ៉ាម៉ែត្រ និងលទ្ធផល
   - ត្រួតពិនិត្យលំនាំប្រើប្រាស់

### 🔑 ភាពខុសគ្នាសំខាន់ៗពី Decorators:

| លក្ខណៈពិសេស | Decorator | Middleware |
|---------|-----------|------------|
| **ដែនកំណត់** | មុខងារតែមួយ | មុខងារទាំងអស់ក្នុងភ្នាក់ងារ |
| **ភាពបត់បែន** | កំណត់នៅពេលកំណត់ | ខ្ពស់នៅពេលរត់ |
| **Context** | មានដែនកំណត់ | មាន context ភ្នាក់ងារពេញលេញ |
| **ការប្រមូលផ្តុំ** | តែមួយឬច្រើន decorators | ជួរដំណើរការមiddleware |
| **ចំណេះដឹងភ្នាក់ងារ** | មិនមាន | មាន (ចូលទៅស្ថានភាពភ្នាក់ងារ) |

### 📚 ពេលណាដែលគួរប្រើ Middleware:

✅ **ប្រើ middleware ពេល៖**
- អ្នកត្រូវការផ្លាស់ប្តូរអាកប្បកិរិយាដោយផ្អែកលើស្ថានភាពអ្នកប្រើ/សម័យ
- អ្នកចង់អនុវត្ត logic ទៅលើមុខងារច្រើន
- អ្នកត្រូវការចូលប្រើ context នៅកម្រិតភ្នាក់ងារ
- អ្នកកំពុងអនុវត្តការព្រួយបារម្ភឆ្លងកាត់ (logging, auth, ល)

❌ **កុំប្រើ middleware ពេល៖**
- ការត្រួតពិនិត្យបញ្ចូលដោយសាមញ្ញ (ប្រើ Pydantic)
- logic ជាក់លាក់មុខងារ (រក្សាទុកក្នុងមុខងារ)
- ការផ្លាស់ប្តូរពេលមួយដង (ផ្លាស់ប្តូរតែម្ដងទេ)

### 🎓 លំនាំកម្រិតខ្ពស់៖

```python
# មេឌៀលវែរខ្វែរ ច្រើន (លំដាប់អនុវត្ដិមានសារៈសំខាន់!)
middleware=[
    logging_middleware,      # កំណត់ហេតុនៅចំពោះមុខ
    auth_middleware,         # បន្ទាប់មកពិនិត្យការផ្ទៀងផ្ទាត់
    cache_middleware,        # បន្ទាប់មកពិនិត្យផ្ទុកបណ្តុះផ្ទុក
    rate_limit_middleware,   # បន្ទាប់មកកំណត់កំណត់ធាតុ
    priority_check_middleware  # ចុងក្រោយពិនិត្យអាទិភាព
]

# អនុវត្ដមេឌៀលវែរខ្វែរ រៀបចំតាមលក្ខខណ្ឌ
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # កែសម្រួលលទ្ធផល
    else:
        # កែចេញអនុវត្ដដោយសាកល្បង
        context.result = cached_value
```

### 🔗 គំនិតដែលពាក់ព័ន្ធ:

- **Agent Middleware**: ចាប់ការហៅ agent.run()
- **Function Middleware**: ចាប់ការហៅមុខងារឧបករណ៍ (អ្វីដែលយើងប្រើ!)
- **Middleware Pipeline**: ជួរដំណើរការមiddleware ត្រូវអនុវត្តតាមលំដាប់
- **Context Propagation**: ផ្ទេរស្ថានភាពតាមជួរដំណើរការមiddleware


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
